In [14]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE

In [15]:
df1 = pd.read_csv('misc/total_2024.csv')
df1['date'] = pd.to_datetime(df1['date'])
df1 = df1.set_index('date').resample('60min').mean().reset_index()
min_date = pd.to_datetime('2024-06-22')
df1 = df1[(df1['date'] >= min_date)]
df1['date'] = pd.to_datetime(df1['date'])
df1 = df1.drop(columns=['10.6',	'10.9',	'11.3',	'11.8',	'12.2',	'12.6',	'13.1',	'13.6',	'14.1'])

In [16]:
df2 = pd.read_csv('misc/2024_met_data.csv')
df2['date'] = pd.to_datetime(df2['date'])
min_date = pd.to_datetime('2024-06-22')
df2 = df2[(df2['date'] >= min_date)]
df2['date'] = pd.to_datetime(df2['date'])
print(df2)

                    date     t    RH       p  SRAD
4151 2024-06-22 00:00:00  24.5  90.0   995.7   0.0
4152 2024-06-22 01:00:00  24.5  91.0   995.5   0.0
4153 2024-06-22 02:00:00  24.6  94.0   994.6   0.0
4154 2024-06-22 03:00:00  24.5  97.0   994.5   0.0
4155 2024-06-22 04:00:00  24.4  97.0   994.2   0.0
...                  ...   ...   ...     ...   ...
8779 2024-12-31 20:00:00   1.5  54.0  1014.6   0.0
8780 2024-12-31 21:00:00   1.5  57.0  1015.3   0.0
8781 2024-12-31 22:00:00   1.2  59.0  1015.3   0.0
8782 2024-12-31 23:00:00   0.7  64.0  1015.3   0.0
8783 2025-01-01 00:00:00   0.5  66.0  1015.5   0.0

[4633 rows x 5 columns]


In [17]:
merged_df = pd.merge(df2, df1, on='date', how='left')
print(merged_df['date'])

0      2024-06-22 00:00:00
1      2024-06-22 01:00:00
2      2024-06-22 02:00:00
3      2024-06-22 03:00:00
4      2024-06-22 04:00:00
               ...        
4628   2024-12-31 20:00:00
4629   2024-12-31 21:00:00
4630   2024-12-31 22:00:00
4631   2024-12-31 23:00:00
4632   2025-01-01 00:00:00
Name: date, Length: 4633, dtype: datetime64[ns]


In [18]:
df = pd.read_csv('smps_output_combined.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.drop(columns=df.columns[df.columns.str.contains('Unnamed')])
df = df.dropna()
print("2. Training data cleaned", flush=True)

# Initial feature/target split
y = df['day.type']
X = df.drop(columns=['day.type', 'date'])

# Apply SMOTE
smote = SMOTE(sampling_strategy='auto')
X, y = smote.fit_resample(X, y)
print(f"3. SMOTE applied. New shape: {X.shape}", flush=True)

2. Training data cleaned
3. SMOTE applied. New shape: (5268, 103)


In [19]:
df = df.drop(columns=['day.type'])

In [20]:
concat_df = pd.concat([df, merged_df])
concat_df = concat_df.dropna()
print(concat_df)

concat_df.to_csv('probe.csv', index=False)

                    date    t    RH       p      SRAD       14.6       15.1  \
0    2016-03-01 17:00:00  4.3  42.0  1018.2  347.2250  24.256917  18.914104   
1    2016-03-01 18:00:00  2.6  47.0  1018.5  144.4456  39.070675  35.658270   
2    2016-03-01 19:00:00  1.4  54.0  1018.9   22.2224  64.734556  66.471501   
3    2016-03-01 20:00:00  0.7  59.0  1019.4    0.0000  54.014671  49.916974   
4    2016-03-01 21:00:00  0.4  61.0  1019.9    0.0000  98.907072  90.041084   
...                  ...  ...   ...     ...       ...        ...        ...   
4628 2024-12-31 20:00:00  1.5  54.0  1014.6    0.0000   6.017999  10.246932   
4629 2024-12-31 21:00:00  1.5  57.0  1015.3    0.0000  10.671358  11.375428   
4630 2024-12-31 22:00:00  1.2  59.0  1015.3    0.0000   8.228971  10.485504   
4631 2024-12-31 23:00:00  0.7  64.0  1015.3    0.0000   8.079387   9.276054   
4632 2025-01-01 00:00:00  0.5  66.0  1015.5    0.0000   3.889335   5.798346   

           15.7       16.3       16.8  ...     358.

In [28]:
df = pd.read_csv('probe.csv')
df['date'] = pd.to_datetime(df['date'])
df2 = pd.read_csv('misc/day.type_h.csv')
df2['date'] = pd.to_datetime(df2['date'])

In [30]:
merged_df = pd.merge(df, df2, on='date', how='left')
merged_df = merged_df.dropna()
print(merged_df)
merged_df.to_csv('probe.csv', index=False)

                    date     t    RH       p      SRAD       14.6       15.1  \
0    2016-03-01 17:00:00   4.3  42.0  1018.2  347.2250  24.256917  18.914104   
1    2016-03-01 18:00:00   2.6  47.0  1018.5  144.4456  39.070675  35.658270   
2    2016-03-01 19:00:00   1.4  54.0  1018.9   22.2224  64.734556  66.471501   
3    2016-03-01 20:00:00   0.7  59.0  1019.4    0.0000  54.014671  49.916974   
4    2016-03-01 21:00:00   0.4  61.0  1019.9    0.0000  98.907072  90.041084   
...                  ...   ...   ...     ...       ...        ...        ...   
5124 2024-10-31 19:00:00  19.0  69.0  1013.4    0.0000  13.314014  15.084722   
5125 2024-10-31 20:00:00  18.4  70.0  1013.9    0.0000  10.057416  12.111346   
5126 2024-10-31 21:00:00  17.3  77.0  1013.9    0.0000   8.378047   9.764409   
5127 2024-10-31 22:00:00  16.6  79.0  1013.6    0.0000   4.672194   6.727176   
5128 2024-10-31 23:00:00  15.7  81.0  1013.9    0.0000   4.754520   5.344365   

           15.7       16.3       16.8  